# DCC 2026 Mission 2 — Stage 13: Untested Model Family First-Screen

이 Notebook은 기존 clean 59D cache와 고정 internal split을 사용해 EBM, Spline-GAM, RFF + Logistic SGD, DCNv2, TabNet, GANDALF를 한 번씩 실험합니다. 저장된 TabM과 Tabular ResNet prediction은 그대로 재사용합니다.

**계약:** train 711,467 / internal validation 177,891, seed 42, call_id overlap 0, training-only scaler, threshold 0.5, Official Validation 사용 금지. Duration/overlap은 정확한 row identity gate 통과 후 오류 분석에만 붙입니다. 모델 설정은 결과를 보기 전에 고정되어 있으며, sweep이나 결과 기반 변경은 없습니다.

실패한 모델은 원인을 기록하고 다음 모델을 계속 실행합니다. 성공하고 checkpoint reload까지 통과한 모델만 통합 성능표에 표시됩니다. 이 first-screen 결과로 최종 모델을 고르거나 ensemble을 만들지 않습니다.


## Checkpoint 0 — Drive와 기존 저장소

Stage 13 출력은 새 폴더에만 저장합니다. 기존 cache, 참조 run, Notebook, experiment log를 수정하지 않습니다.


In [ ]:
from pathlib import Path
import sys, os, json, time, platform, importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display, Markdown
from google.colab import drive

drive.mount('/content/drive', force_remount=True)
REPO_ROOT = Path('/content/drive/MyDrive/prelim')
assert (REPO_ROOT / 'src/dcc_data/mission2_model_family_screen.py').is_file(), REPO_ROOT
sys.path.insert(0, str(REPO_ROOT / 'src'))
print('Repository:', REPO_ROOT)
print('Stage 13 output root: data/processed/mission2_stage13_untested_model_family_first_screen/')


## Checkpoint A — 모델 라이브러리 준비

Colab 기본 PyTorch/CUDA 설치는 유지합니다. 아래 패키지는 각 모델 구현에 필요한 라이브러리이며, 공식 validation이나 데이터 추출을 실행하지 않습니다.


In [ ]:
%pip -q install interpret==0.6.10 pytorch-tabnet==4.1.0 pytorch-tabular==1.1.1 pyarrow


In [ ]:
import torch, sklearn
print({'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
       'scikit_learn': sklearn.__version__, 'torch': torch.__version__,
       'cuda_available': torch.cuda.is_available(),
       'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})


## Checkpoint B — Clean cache, split, label, scaler identity

기존 loader가 source fingerprint와 59개 feature 순서, split, call 분리를 검사합니다. 이 공통 gate가 실패하면 여기서 중단합니다. 기존 feature cache를 읽을 뿐, 추출이나 waveform 처리는 하지 않습니다.


In [ ]:
from dcc_data import mission2_model_family_screen as screen
from dcc_data import mission2_stage13_untested_model_family as stage13

context = screen.load_screen(REPO_ROOT)
contract = stage13._assert_fixed_contract(context)
print(json.dumps({'source_fingerprint': contract['provenance_fingerprint'],
                  'feature_dim': contract['feature_dim'],
                  'train_rows': len(context.prepared.train_idx),
                  'validation_rows': len(context.prepared.validation_idx),
                  'train_calls': len(np.unique(context.train_calls)),
                  'validation_calls': len(np.unique(context.validation_calls)),
                  'call_overlap': len(set(context.train_calls) & set(context.validation_calls)),
                  'seed': contract['seed'], 'threshold': contract['threshold'],
                  'scaler_fit_scope': contract['scaler_fit_scope'],
                  'official_validation_used': contract['official_validation_used']}, indent=2))


## Checkpoint C — Frozen reference와 분석용 annotation gate

TabM/Tabular ResNet은 저장된 prediction과 metrics만 사용합니다. scaler 통계, cache/split/label contract, row identity를 재검증하고 metric을 prediction에서 다시 계산합니다. Duration·overlap은 manifest를 `record_key`로 일대일 조인한 뒤 `call_id`와 label까지 확인하며 feature matrix에는 넣지 않습니다.


In [ ]:
tracking = stage13._load_tracking(REPO_ROOT, context)
references = stage13.load_frozen_references(REPO_ROOT, context)
print('Frozen references:', references['root'])
print('Exact tracking rows:', len(tracking), '| duration range (s):',
      (float(tracking.duration_seconds.min()), float(tracking.duration_seconds.max())))
print('Overlap rows (analysis only):', int(tracking.overlap.sum()))
display(pd.DataFrame([{'model': name, 'accuracy': m['validation_accuracy'],
                       'recall speaker 0': m['recall_0'], 'recall speaker 1': m['recall_1'],
                       'HCE >= .90': m['hce'], 'checkpoint reload': m['checkpoint_reload']}
                      for name, m in references['metrics'].items()]))


## Checkpoint D — 설정 고정과 새 run 예약

아래 여섯 설정을 그대로 기록하고 각 모델을 한 번만 학습합니다. 신경망 후보는 CUDA가 있으면 사용하고, 없으면 CPU에서 시도하며 그 환경 여부와 오류도 기록합니다. 모델별 결과에 따라 설정을 바꾸지 않습니다.


In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
environment = {'python': platform.python_version(), 'platform': platform.platform(),
    'numpy': np.__version__, 'pandas': pd.__version__, 'scikit_learn': sklearn.__version__,
    'torch': torch.__version__, 'cuda_available': torch.cuda.is_available(), 'device': DEVICE,
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'packages': {name: importlib.metadata.version(name) for name in
       ['interpret', 'pytorch-tabnet', 'pytorch-tabular', 'pyarrow']}}
run_root = stage13._new_run(REPO_ROOT, context, references, tracking, environment)
print('Frozen config IDs:')
display(pd.DataFrame([{'model': stage13.MODEL_LABELS[name], **stage13.CONFIGS[name]}
                      for name in stage13.MODEL_IDS]))
print('Unique Stage 13 run:', run_root)

run_results = {}
def checkpoint_result(model_id):
    result = stage13.run_candidate(model_id, context, run_root, device=DEVICE)
    run_results[model_id] = result
    display(Markdown('### 실행 결과 — ' + stage13.MODEL_LABELS[model_id]))
    if result.get('status') != 'SUCCESS':
        display(Markdown('**관찰:** 이 모델은 학습 또는 reload 검증에서 실패했습니다. 성능 수치는 만들지 않았고, 다음 모델은 계속 실행합니다.'))
        print(json.dumps(result, ensure_ascii=False, indent=2))
        return
    frame = pd.read_parquet(run_root / model_id / 'predictions.parquet')
    compare_rows = []
    for name in ('tabm', 'tabular_resnet', model_id):
        m = references['metrics'][name] if name in references['metrics'] else result
        compare_rows.append({'모델': stage13.MODEL_LABELS[name], 'Accuracy': m['validation_accuracy'],
            'Train−Val gap': m['train_val_gap'], 'Recall 0': m['recall_0'], 'Recall 1': m['recall_1'],
            'TN': m['confusion_matrix'][0][0], 'FP': m['confusion_matrix'][0][1],
            'FN': m['confusion_matrix'][1][0], 'TP': m['confusion_matrix'][1][1],
            'HCE ≥ 0.90': m['hce'], 'runtime (s)': m.get('runtime_seconds', m.get('fit_seconds',0)+m.get('predict_seconds',0)),
            'reload': m['checkpoint_reload']})
    display(pd.DataFrame(compare_rows).set_index('모델'))
    base_a = references['predictions']['tabm'].correct.to_numpy(bool)
    base_b = references['predictions']['tabular_resnet'].correct.to_numpy(bool)
    candidate = frame.correct.to_numpy(bool)
    common = ~base_a & ~base_b
    corrected = int((common & candidate).sum())
    regressed_tabm = int((base_a & ~candidate).sum())
    fixed_tabm = int((~base_a & candidate).sum())
    print(f"혼동행렬 [[TN, FP], [FN, TP]] = {result['confusion_matrix']}")
    print(f"기존 두 reference의 공통 오류 {int(common.sum())}건 중 교정 {corrected}건")
    print(f"TabM 비교: 새 모델만 정답 {fixed_tabm}건, TabM만 정답 {regressed_tabm}건; net={fixed_tabm-regressed_tabm:+d}건")
    print(f"관찰: Accuracy 변화는 TabM 대비 {(result['validation_accuracy']-references['metrics']['tabm']['validation_accuracy'])*100:+.3f}%p, "
          f"speaker 0/1 Recall은 {result['recall_0']:.4f}/{result['recall_1']:.4f}, "
          f"HCE는 {result['hce']}건, Train−Val gap은 {result['train_val_gap']:.4f}입니다.")
    print('가능한 원인: 이 first-screen은 구조와 학습 recipe가 함께 달라지므로, 관찰된 차이를 구조 하나의 인과 효과로 단정할 수 없습니다. Accuracy, class별 Recall, FP/FN, HCE와 paired 교정/회귀를 함께 읽습니다.')


## Checkpoint E — EBM

**무엇을 하는가:** 각 feature 효과를 additive하게 학습하는 Explainable Boosting Machine입니다. 이번에는 feature 간 interaction을 끄고 59D 음향 표현의 비선형 단변량 효과가 reference와 다른 오류를 교정하는지 봅니다. 학습은 고정 100 rounds이며 validation으로 설정을 바꾸지 않습니다.


In [ ]:
checkpoint_result('ebm')


## Checkpoint F — Spline-GAM

**무엇을 하는가:** 각 feature를 training quantile knots의 cubic spline basis로 바꾼 뒤 logistic regression을 적용하는 additive GAM입니다. EBM과 다른 부드러운 함수 형태의 first-screen을 비교합니다. spline knots는 train에서만 fit하고 validation은 변환만 합니다.


In [ ]:
checkpoint_result('spline_gam')


## Checkpoint G — RFF + Logistic SGD

**무엇을 하는가:** 고정 seed의 Random Fourier Features로 RBF kernel의 비선형 유사도를 근사하고, logistic SGD를 batch 단위로 10회 학습합니다. 대규모 데이터에서 batch 학습이 잘 맞는지, reference 오류 중 무엇을 교정하는지 확인합니다. validation은 학습 update에 넣지 않습니다.


In [ ]:
checkpoint_result('rff_logistic_sgd')


## Checkpoint H — DCNv2

**무엇을 하는가:** DCNv2 형태의 cross layer가 feature 조합을 명시적으로 만들고, 병렬 MLP가 다른 비선형 패턴을 학습합니다. 2개 cross layer와 `[128, 64]` MLP, 최대 20 epoch, 내부 validation patience 5를 사용하며 best validation accuracy checkpoint를 저장합니다.


In [ ]:
checkpoint_result('dcnv2')


## Checkpoint I — TabNet

**무엇을 하는가:** 여러 decision step에서 feature attention mask를 학습하는 tabular neural model입니다. 선택적 attention이 같은 59D에서 reference의 오류를 달리 처리하는지 봅니다. 최대 20 epoch, validation accuracy patience 5입니다.


In [ ]:
checkpoint_result('tabnet')


## Checkpoint J — GANDALF

**무엇을 하는가:** Gated Feature Learning Unit을 쌓아 feature 간 관계를 게이트 형태로 학습하는 tabular neural model입니다. 최대 20 epoch와 validation accuracy patience 5를 사용합니다. 구현/API가 현재 runtime과 맞지 않으면 원인을 남기고 다른 결과는 계속 보존합니다.


In [ ]:
checkpoint_result('gandalf')


## Checkpoint K — 8개 모델 통합 성능과 혼동행렬

여섯 신규 후보 중 실제 학습·평가·reload가 모두 성공한 모델만 reference 두 개와 함께 표시됩니다. 따라서 실패가 있으면 비교 행 수가 8보다 작으며, 그 결과를 성공한 것처럼 채우지 않습니다.


In [ ]:
comparison, paired, review = stage13.finalize_run(run_root, context, references, tracking)
comparison = comparison.sort_values('accuracy', ascending=False).reset_index(drop=True)
display(comparison)
print('Status summary:')
print(json.dumps(json.loads((run_root/'run_summary.json').read_text()), ensure_ascii=False, indent=2))


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
comparison.plot.bar(x='model', y='accuracy', ax=axes[0,0], legend=False, color='#31688e')
axes[0,0].set_title('Internal Validation Accuracy'); axes[0,0].tick_params(axis='x', rotation=35)
comparison.plot.bar(x='model', y='train_minus_val_gap', ax=axes[0,1], legend=False, color='#b35c44')
axes[0,1].set_title('Train − Validation Accuracy gap'); axes[0,1].tick_params(axis='x', rotation=35)
comparison.set_index('model')[['recall_speaker_0','recall_speaker_1']].plot.bar(ax=axes[1,0])
axes[1,0].set_title('Recall by speaker'); axes[1,0].tick_params(axis='x', rotation=35)
comparison.plot.bar(x='model', y='HCE_ge_0_90', ax=axes[1,1], legend=False, color='#7b3294')
axes[1,1].set_title('High-confidence errors (confidence ≥ 0.90)'); axes[1,1].tick_params(axis='x', rotation=35)
for ax in axes.flat: ax.grid(axis='y', alpha=.25)
plt.tight_layout(); plt.show()


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, row in zip(axes.flat, comparison.to_dict('records')):
    cm = np.array([[row['TN'], row['FP']], [row['FN'], row['TP']]])
    image = ax.imshow(cm, cmap='Blues')
    ax.set_title(row['model']); ax.set_xticks([0,1], ['predict 0','predict 1']); ax.set_yticks([0,1], ['true 0','true 1'])
    for (i,j), value in np.ndenumerate(cm): ax.text(j,i,f'{value:,}',ha='center',va='center')
fig.colorbar(image, ax=axes.ravel().tolist(), shrink=.75, label='utterance count')
fig.suptitle('Confusion matrices — same internal validation rows and threshold 0.5')
plt.show()


## Checkpoint L — paired 오류, 공통 오류, 취약 구간, confidence

Paired 비교는 동일 validation `record_key` 행끼리 reference 정답/오답이 candidate에서 어떻게 바뀌었는지 셉니다. Duration/overlap은 모델 feature가 아니라 분석용 annotation입니다. `n ≥ 100` slice를 우선 읽고, 작은 slice는 우연 변동 가능성이 커서 단정하지 않습니다.


In [ ]:
paired = pd.read_csv(run_root/'paired_error_comparison.csv')
common = pd.read_csv(run_root/'common_reference_error_corrections.csv')
slices = pd.read_csv(run_root/'slice_comparison.csv')
class_recall = pd.read_csv(run_root/'class_recall_comparison.csv')
print('Candidate vs each frozen reference:')
display(paired)
print('Both references wrong: corrected vs still wrong:')
display(common)
print('Full-validation speaker recall:')
display(class_recall.pivot(index='model', columns='speaker', values='recall').rename(columns={0:'speaker 0 recall',1:'speaker 1 recall'}))
print('Duration / overlap / class / confidence slices with at least 100 rows:')
display(slices[(slices.n >= 100) & (slices.slice_type != 'confidence')].sort_values(['slice_type','slice','accuracy']))
print('Confidence bins and high-confidence error counts:')
display(slices[slices.slice_type == 'confidence'].sort_values(['model','slice']))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for slice_type, ax in [('duration',axes[0]),('overlap',axes[1])]:
    part = slices[(slices.slice_type == slice_type) & (slices.n >= 100)]
    if not part.empty:
        part.pivot(index='slice', columns='model', values='accuracy').plot.bar(ax=ax)
    ax.set_title(f'{slice_type} slice accuracy (n ≥ 100)'); ax.set_ylabel('accuracy'); ax.grid(axis='y',alpha=.25)
    ax.tick_params(axis='x',rotation=25)
plt.tight_layout(); plt.show()


In [ ]:
print('Samples where at least one model is wrong:', len(review))
review_columns = ['record_key','call_id','y_true','duration_seconds','overlap','audio_path','annotation_row']
review_columns += [f'{m}__prediction' for m in comparison.model_id]
review_columns += [f'{m}__confidence' for m in comparison.model_id]
display(review[review_columns].head(20))
hce_cases = pd.read_parquet(run_root/'high_confidence_error_cases.parquet')
unique_cases = pd.read_parquet(run_root/'model_unique_correction_cases.parquet')
common_cases = pd.read_parquet(run_root/'all_model_common_errors.parquet')
print('Rows with any model HCE:', len(hce_cases))
print('Rows uniquely corrected by exactly one of the compared models:', len(unique_cases))
print('Rows wrong for every compared model:', len(common_cases))
print('Review artifacts:', run_root/'misclassified_sample_review.parquet',
      run_root/'high_confidence_error_cases.parquet', run_root/'model_unique_correction_cases.parquet')


## Checkpoint M — 초보자용 종합 해석과 한계

다음 요약은 관측된 수치만 계산합니다. 관찰과 가능한 설명을 분리하며, best accuracy 하나만으로 모델을 선정하지 않습니다.


In [ ]:
tabm = comparison.loc[comparison.model_id == 'tabm'].iloc[0]
resnet = comparison.loc[comparison.model_id == 'tabular_resnet'].iloc[0]
new_rows = comparison[~comparison.model_id.isin(['tabm','tabular_resnet'])]
print(f"관찰된 모델 수: {len(comparison)}개 (frozen reference 2개, 성공한 신규 후보 {len(new_rows)}개).")
if len(new_rows):
    best = new_rows.loc[new_rows.accuracy.idxmax()]
    print(f"신규 후보 중 Accuracy가 가장 높았던 행은 {best['model']} ({best.accuracy:.4f})이며, TabM 대비 {(best.accuracy-tabm.accuracy)*100:+.3f}%p, ResNet 대비 {(best.accuracy-resnet.accuracy)*100:+.3f}%p입니다. 이는 first-screen 비교이지 최종 선택이 아닙니다.")
    for row in new_rows.to_dict('records'):
        delta0 = row['recall_speaker_0']-tabm['recall_speaker_0']
        delta1 = row['recall_speaker_1']-tabm['recall_speaker_1']
        print(f"- {row['model']}: accuracy {row['accuracy']:.4f} ({(row['accuracy']-tabm['accuracy'])*100:+.3f}%p vs TabM), speaker0/1 Recall {row['recall_speaker_0']:.4f}/{row['recall_speaker_1']:.4f} (변화 {delta0:+.4f}/{delta1:+.4f}), HCE {int(row['HCE_ge_0_90'])}, gap {row['train_minus_val_gap']:.4f}, runtime {row['runtime_seconds']:.1f}s.")
    print('관찰: paired 교정과 회귀는 Checkpoint L 표를 함께 보세요. Accuracy가 비슷해도 FP/FN, class recall, HCE, 취약 slice에서 오류 구성이 다를 수 있습니다.')
    print('가능한 원인: 모델의 함수 형태와 학습 recipe가 함께 달라졌습니다. 이 결과만으로 특정 구조가 차이를 일으켰다고 말할 수 없고, slice 성능은 표본 수와 row identity를 함께 확인해야 합니다.')
else:
    print('신규 모델 중 성공·reload 검증을 통과한 결과가 없습니다. 미실행 성능을 만들지 않았습니다.')
print('미해결/제약: 이 Notebook은 고정 recipe의 first-screen입니다. 일부 package/backend 실패, 수렴 한계, GPU/runtime 차이가 있으면 해당 failure.json과 environment.json을 확인하세요. 공식 validation은 사용하지 않았습니다. Stage 14, ensemble, final model selection은 실행하지 않았습니다.')
print('Run folder:', run_root)


In [ ]:
# Google Colab UI에서 Ctrl+S를 눌러 실행 output이 포함된 Notebook을 Drive에 저장합니다.
print('Notebook checkpoint outputs are visible above. Save this Colab Notebook in the UI now.')
print('Run artifact summary:', (run_root/'run_summary.json').read_text())
